In [34]:
!pip install pandasql

In [35]:
import pandas as pd
from pandasql import sqldf

In [36]:
pysqldf = lambda q: sqldf(q, globals())

In [37]:
buyers = pd.read_csv('/content/buyers.csv', sep=',')
order_items = pd.read_csv('/content/order_items.csv', sep=',')
orders = pd.read_csv('/content/orders.csv', sep=',')
payments = pd.read_csv('/content/payments.csv', sep=',')
products = pd.read_csv('/content/products.csv', sep=',')
sellers = pd.read_csv('/content/sellers.csv', sep=',')

In [38]:
# display(buyers.head())
# display(order_items.head())
# display(orders.head())
# display(payments.head())
# display(products.head())
# display(sellers.head())


# Desafio 1

Nesse exercício considerei a data 29/11/2024 como o dia atual, pois é o maior created_at da base de dados. Em uma situação real, ao invés de '2024-11-29' deveria ser utilizado 'now'.

1.   Foi necessário realizar um filtro de status e um filtro de data de criação, considerando apenas 12 meses a partir do dia 1 (parâmetro 'start of month') do mês atual.
2.   Em seguida, realizei um agrupamento baseado no campo ano/mês para calcular o faturamento bruto mensal, a quantidade de pedidos e o valor do ticket médio.
3.   Por fim, ordenei de maneira decrescente considerando o campo ano/mês.

In [39]:
# Desafio 1
query = """
SELECT
  STRFTIME('%Y/%m', created_at) AS mes,
  SUM(total_value) AS faturamento_bruto_mensal,
  COUNT(id) AS qtd_pedidos,
  ROUND(AVG(total_value), 2) AS ticket_medio
FROM orders
WHERE status IN ('completed', 'delivered')
AND created_at >= DATE('2024-11-29', 'start of month', '-12 months')
GROUP BY 1
ORDER BY 1 DESC
"""

resultado_desafio1 = pysqldf(query)
resultado_desafio1


,mes,faturamento_bruto_mensal,qtd_pedidos,ticket_medio
0,2024/11,56710238.63,3643,15566.91
1,2024/10,62493439.13,3863,16177.44
2,2024/09,60274462.50,3779,15949.84
3,2024/08,60102539.82,3743,16057.32
4,2024/07,59769406.95,3862,15476.28
5,2024/06,58797774.37,3644,16135.50
6,2024/05,60730315.72,3848,15782.31
7,2024/04,57779312.59,3653,15816.95
8,2024/03,61525913.64,3877,15869.46
9,2024/02,57926376.26,3628,15966.48


# Desafio 2

Novamente considerei a data 29/11/2024 como o dia atual por ser a maior data da base de dados, mas em uma situação real em todo lugar onde aparece '2024-11-29' deveria ser utilizado 'now'.

1.   Primeiramente criei a CTE "trimestres" para obter o trimestre atual e o trimestre anterior.
2.   Em seguinda, na CTE "pedidos_por_trimestre" filtrei os últimos 6 meses, pois é o máximo de dados que preciso para trabalhar com os últimos dois semestres, e agrupei esses dados por semestre e por vendedor, para ser possível calcular o GMV de cada vendedor por semestre.
3.   As CTEs "trimestre_atual" e "trimestre_anterior" separam os dados da "pedidos_por_trimestre" em vendas do trimestre atual e vendas do trimestre anterior.
4.   O SELECT final filtra apenas os vendedores com mais de 50 vendas em ambos os trimestres, calcula a porcentagem de crescimento de cada vendedor entre o trimestre anterior e o trimestre atual.
5.   Por fim, ordenei em ordem descrescente a partir das porcentagens calculadas.

Os valores negativos indicam que as vendas diminuíram no trimestre atual em relação ao trimestre anterior.

In [40]:
# Desafio 2
query = """
WITH trimestres AS (
  SELECT
    trimestre_atual,
    ano_trimestre_atual,
    CASE
      WHEN trimestre_atual = 1 THEN 4
      ELSE trimestre_atual - 1
    END AS trimestre_anterior,
    CASE
      WHEN trimestre_atual = 1 THEN ano_trimestre_atual - 1
      ELSE ano_trimestre_atual
    END AS ano_trimestre_anterior
    FROM (
      SELECT
        (((STRFTIME('%m', '2024-11-29') - 1) / 3) + 1) AS trimestre_atual,
        STRFTIME('%Y', '2024-11-29') AS ano_trimestre_atual
    )
),
pedidos_por_trimestre AS (
  SELECT
    seller_id,
    (((STRFTIME('%m', created_at) - 1) / 3) + 1) AS trimestre,
    STRFTIME('%Y', created_at) AS ano,
    SUM(total_value) AS gmv,
    COUNT(id) AS qtd_pedidos
  FROM orders
  WHERE created_at >= DATE('2024-11-29', 'start of month', '-6 months')
  GROUP BY 1, 2, 3
),
trimestre_atual AS (
  SELECT ppt.*
  FROM pedidos_por_trimestre ppt
    INNER JOIN trimestres t ON ppt.ano = t.ano_trimestre_atual
    AND ppt.trimestre = t.trimestre_atual
),
trimestre_anterior AS (
  SELECT ppt.*
  FROM pedidos_por_trimestre ppt
    INNER JOIN trimestres t ON ppt.ano = t.ano_trimestre_anterior
    AND ppt.trimestre = t.trimestre_anterior
)
SELECT
  s.name AS vendedor,
  s.state AS estado,
  tatu.gmv AS gmv_trimestre_atual,
  tant.gmv AS gmv_trimestre_anterior,
  ROUND((tatu.gmv - tant.gmv) / tant.gmv * 100, 2) AS perc_crescimento
FROM sellers s
  INNER JOIN trimestre_atual tatu ON tatu.seller_id = s.id
  INNER JOIN trimestre_anterior tant ON tant.seller_id = s.id
WHERE tatu.qtd_pedidos >= 50
  AND tant.qtd_pedidos >= 50
ORDER BY 5 DESC
LIMIT 10;
"""

resultado_desafio2 = pysqldf(query)
resultado_desafio2

,vendedor,estado,gmv_trimestre_atual,gmv_trimestre_anterior,perc_crescimento
0,Costa & Ferreira Atacado Distribuidora,PE,970702.77,977776.85,-0.72
1,Santos & Silva Suprimentos Distribuidora,MS,1116094.58,1186919.92,-5.97
2,Rodrigues & Almeida Atacado Distribuidora,MG,1108548.84,1185897.83,-6.52
3,Costa & Silva Alimentos Distribuidora,BA,1265522.36,1381962.73,-8.43
4,Santos & Silva Mercado Distribuidora,MT,1126498.27,1244600.51,-9.49
5,Nascimento & Souza Alimentos Distribuidora,GO,1042286.03,1156454.52,-9.87
6,Nascimento & Souza Alimentos Distribuidora,SP,1207319.80,1342900.90,-10.10
7,Rodrigues & Oliveira Atacado Distribuidora,RS,979175.80,1100711.68,-11.04
8,Costa & Santos Alimentos Distribuidora,MG,1051866.85,1188636.29,-11.51
9,Lima & Almeida Grupo Distribuidora,CE,1171211.61,1328518.10,-11.84


# Desafio 3

Após análise nas bases, observei que o desconto não está aplicado ao valor unitário (unit_price) da tabela order_items. Portanto, através desse campo consegui calcular o valor total do pedido sem desconto.
1.   A CTE "perc_desconto" filtra os pedidos cancelados e agrupa apartir do pedido e do vendedor para calcular o total de desconto, total bruto e percentual de desconto.
2.  O último SELECT apresenta as vendas realizadas com mais de 40% de desconto e exibe qual o vendedor responsável pela venda e as outras informações solicitadas.

In [41]:
# Desafio 3
query = """
WITH perc_desconto AS (
  SELECT
    oi.order_id,
    o.seller_id,
    o.created_at,
    SUM(oi.discount) AS total_desconto,
    SUM(oi.unit_price * oi.qty) AS total_bruto,
    o.total_value AS total_liquido,
    ROUND(SUM(oi.discount) / SUM(oi.unit_price * oi.qty) * 100, 2) AS perc_desconto
  FROM orders o
    INNER JOIN order_items oi ON oi.order_id = o.id
  WHERE o.status NOT IN ('cancelled')
  GROUP BY 1, 2, 3, 6
)
SELECT
  s.name AS vendedor,
  pd.order_id,
  pd.total_bruto,
  pd.total_desconto,
  pd.total_liquido,
  pd.perc_desconto,
  pd.created_at AS data_venda
FROM perc_desconto pd
  INNER JOIN sellers s ON s.id = pd.seller_id
WHERE perc_desconto > 40
"""

resultado_desafio3 = pysqldf(query)
resultado_desafio3

,vendedor,order_id,total_bruto,total_desconto,total_liquido,perc_desconto,data_venda
0,Costa & Costa Atacado Distribuidora,274,13611.57,6781.87,6829.70,49.82,2024-11-02 03:11:05
1,Costa & Costa Atacado Distribuidora,490,41396.27,17992.05,23404.22,43.46,2023-10-13 04:08:32
2,Costa & Costa Atacado Distribuidora,602,10494.05,4712.86,5781.19,44.91,2023-08-27 08:58:16
3,Costa & Costa Atacado Distribuidora,879,4500.99,2596.89,1904.10,57.70,2023-12-09 03:59:32
4,Costa & Costa Atacado Distribuidora,994,21286.75,9310.53,11976.22,43.74,2024-07-31 17:45:23
...,...,...,...,...,...,...,...
895,Costa & Oliveira Atacado Distribuidora,75858,12004.41,7026.07,4978.34,58.53,2023-12-21 01:01:25
896,Costa & Oliveira Atacado Distribuidora,76494,22937.44,10333.95,12603.49,45.05,2024-01-01 17:31:54
897,Costa & Oliveira Atacado Distribuidora,77160,8856.92,4219.58,4637.34,47.64,2024-10-07 04:45:13
898,Costa & Oliveira Atacado Distribuidora,77815,3124.52,1429.18,1695.34,45.74,2024-07-28 05:34:09


# Desafio 4

Nesse exercício utilizei a função RANK para garantir que, se dois produtos tiverem o mesmo preço, ambos receberão o rank = 1, pois ambos tiveram o maior preço do pedido.

1.   A CTE "rank_produtos_por_preco" númera, particionando por pedido, os produtos com maior valor unitário de cada venda realizada.
2.   A CTE "produtos_mais_caros" filtra os produtos mais caros de cada uma das vendas.
3.   A CTE "qtd_vendida" calcula a quantidade vendida de cada produto em todas as vendas realizadas.
4.   O último SELECT apresenta todos os produtos com mais de 1000 vendas na história da empresa, mas que nunca foram o produto mais caro de pelo menos um pedido.

Como nenhum resultado foi retornado, significa que todos os produtos que tem mais de 1000 vendas já foram o produto com maior valor unitário de pelo menos um pedido.

In [42]:
# Desafio 4
query = """
WITH rank_produtos_por_preco AS (
  SELECT
    order_id,
    product_id,
    RANK() OVER (
      PARTITION BY order_id
      ORDER BY unit_price DESC
    ) AS rank
  FROM order_items
),
produtos_mais_caros AS (
  SELECT DISTINCT
    product_id
  FROM rank_produtos_por_preco
  WHERE rank = 1
),
qtd_vendida AS (
  SELECT
    product_id,
    SUM(qty) AS qtd_total_vendida
  FROM order_items
  GROUP BY 1
)
SELECT DISTINCT
  p.name AS produto,
  p.category AS categoria,
  qv.qtd_total_vendida
FROM qtd_vendida qv
  INNER JOIN products p ON p.id = qv.product_id
  LEFT JOIN produtos_mais_caros pcm ON pcm.product_id = qv.product_id
WHERE qv.qtd_total_vendida > 1000
  AND pcm.product_id IS NULL
"""

resultado_desafio4 = pysqldf(query)
resultado_desafio4

,produto,categoria,qtd_total_vendida
